# COMP5122M – Data Science Coursework

## Project: Vibe Check - How Gen Z Really Feels About AI


## Team - Shyamin 

### Members: 

Durgam Naga Deeraj Reddy (pvcl0430) ; Harshini Gonela (jklt0901) ; Karuna Anil Gholap (lctl0641) ; Rehan Ali (fnkx0448)



## Group Data Science Analysis

## Project Overview
This project analyses the Pew Research Center's Spring 2025 Global Attitudes Survey to investigate public attitudes towards artificial intelligence across 24 countries.

Task 1 focuses on data quality assessment and preprocessing. The objectives are to inspect the original respondent-level dataset, identify and address missing and invalid values, correctly interpret survey response codes, harmonise education categories across countries, and prepare a reliable dataset for subsequent statistical analysis. The preprocessing workflow preserves the original survey responses, documents all transformation decisions, and verifies the quality of the resulting analytical dataset.

## Task 1: Data Quality and Preprocessing

Performed by Durgam Naga Deeraj Reddy (pvcl0430)

### Importing Required Libraries

In [5]:
import pandas as pd
import numpy as np
import re

### Load the csv file

In [6]:
# load the original pew dataset and data dictionary
df = pd.read_csv("/Users/sdreddy/Leeds/Sem 1/Data Science/coursework 1/Resources/Pew_Original_Data/Pew Research Center Global Attitudes Spring 2025 Dataset CSV.csv", low_memory=False)

print("Original dataset dimensions:", df.shape)


Original dataset dimensions: (28333, 483)


### Check the dataset information and make a brief

In [7]:
print("Dataset dimensions (rows, columns):", df.shape)

Dataset dimensions (rows, columns): (28333, 483)


### Check for duplicates

In [8]:
# if there's no print, it'll return duplicates sum along with their respective data type
duplicate_count = df.duplicated().sum()
print("Completely duplicated rows:", duplicate_count)   # print statement gives the sum value of duplicates

Completely duplicated rows: 0


In [9]:
# a respondednt ID is expected to be unique within each country
# if the dataset contains repeated IDs within a country, it may indicate data entry errors or other issues that need to be addressed. So we have to check for duplicate respondent IDs within countries
if {"country", "id"}.issubset(df.columns):
    duplicate_id_mask = df.duplicated(subset=["country", "id"], keep=False)     # a check for duplicate respondent IDs within countries
    print("Rows involved in repeated country-ID combinations:", duplicate_id_mask.sum()) 
    print("Repeated country-ID combinations:", df.loc[duplicate_id_mask, ["country", "id"]].drop_duplicates().shape[0])  
    display(df.loc[duplicate_id_mask].sort_values(["country", "id"]).head(20))
else:
    print("The country and/or id column is not present; duplicate-ID check skipped.") 
# anyhow respondednt ID's are not duplicated, but even if they were, we would have to decide how to handle them
# before removing the duplicates we should investigate further to determine the cause of the duplication and take appropriate action

Rows involved in repeated country-ID combinations: 0
Repeated country-ID combinations: 0


,survey,country,weight,id,econ_sit,satisfied_democracy,polsys_satisfied,fav_us,fav_china,fav_eu,...,region_skorea,region_spain,region_sweden,region_turkey,region_uk,urbanicity,qdate_s,qdate_e,intdate_aus,phone_sample


### Check for missing values

In [10]:
missing_before_whitespace = df.isna().sum()
print("Missing cells recognised by pandas before whitespace cleaning:", int(missing_before_whitespace.sum()))
# here it shows 0 missing values, but on inspection, we can see that there are missing values in the dataset
# this is because the missing values are represented as empty strings or other placeholders, which are not recognized as NaN by pandas
# to handle this, we can convert these whitespaces with NaN and then check for missing values again
# in the output we are getting the missing values after cleaning the whitespaces because after cleaning the whitespaces, the missing values are now recognized as NaN by pandas, and we can see the correct count of missing values in the dataset

Missing cells recognised by pandas before whitespace cleaning: 0


In [11]:
df = df.replace(r"^\s*$", np.nan, regex=True).infer_objects(copy=False)   # this command converts all empty strings to NaN values in the DataFrame
# lets create a table to understand which variables have missing values before and after whitespace cleaning, and how many new missing values were identified as a result of the cleaning process
missing_after_whitespace = df.isna().sum()
missing_summary = pd.DataFrame({"missing_before_whitespace_cleaning": missing_before_whitespace, "missing_after_whitespace_cleaning": missing_after_whitespace})
missing_summary["newly_identified_missing"] = (missing_summary["missing_after_whitespace_cleaning"] - missing_summary["missing_before_whitespace_cleaning"])
print("Total missing cells after whitespace cleaning:", int(missing_after_whitespace.sum()))
display(missing_summary.sort_values("missing_after_whitespace_cleaning", ascending=False).head(30)) 

Total missing cells after whitespace cleaning: 10653971


/var/folders/15/vnzgb5095kj14qn7m2szdjr80000gn/T/ipykernel_15237/2552235401.py:1: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df = df.replace(r"^\s*$", np.nan, regex=True).infer_objects(copy=False)   # this command converts all empty strings to NaN values in the DataFrame


,missing_before_whitespace_cleaning,missing_after_whitespace_cleaning,newly_identified_missing
d_age_argentina,0,28333,28333
d_age_kenya,0,28333,28333
d_ptylean_kenya,0,28333,28333
d_age_brazil,0,28333,28333
d_age_skorea,0,28333,28333
d_age_safrica,0,28333,28333
d_age_mexico,0,28333,28333
d_age_indonesia,0,28333,28333
d_age_nigeria,0,28332,28332
d_age_sweden,0,28332,28332


### Special response codes

Survey response codes such as `8`, `9`, `98` and `99` can represent different things in different variables. Use the data dictionary to identify them variable by variable. In particular, `age` code 98 is Don't know and code 99 is Refused; code 97 (97 or older) is valid. For Q32 trust variables, code 8 (Not sure) is a valid response and must be retained.


The dictionary is loaded in the next code cell. Apostrophe variants in its labels are normalised before identifying variable-specific **Don't know** and **Refused** codes. This prevents the curly apostrophe in the age label from being missed.


In [12]:
# load the official Values sheet from the pew data dictionary
values_dictionary = pd.read_excel(
    "../Resources/Pew_Original_Data/Pew Research Center Global Attitudes Spring 2025 Data Dictionary.xlsx",
    sheet_name="Values",
    header=None
)
values_dictionary = values_dictionary.iloc[2:].copy()
values_dictionary.columns = ["variable", "value_code", "value_label"]
values_dictionary["variable"] = values_dictionary["variable"].ffill()
values_dictionary = values_dictionary.dropna(
    subset=["variable", "value_code", "value_label"]
).copy()

# normalise straight and curly apostrophes before searching labels
values_dictionary["value_label"] = (
    values_dictionary["value_label"].astype(str)
    .str.replace("’", "'", regex=False)
    .str.replace("‘", "'", regex=False)
    .str.strip()
)

# identify non-response codes by variable and do not recode numeric values globally
special_codes = values_dictionary[
    values_dictionary["value_label"].str.contains(
        r"\bdon't\s+know\b|\brefused\b",
        case=False, regex=True, na=False
    )
].copy()
special_codes["value_code"] = pd.to_numeric(
    special_codes["value_code"], errors="coerce"
)
special_codes = special_codes.dropna(
    subset=["variable", "value_code"]
).drop_duplicates(subset=["variable", "value_code"])

print("Variable-specific Don't know / Refused codes found:", len(special_codes))
display(special_codes.head(25))


Variable-specific Don't know / Refused codes found: 879


,variable,value_code,value_label
31,econ_sit,8.0,Don't know (DO NOT READ)
32,econ_sit,9.0,Refused (DO NOT READ)
37,satisfied_democracy,8.0,Don't know (DO NOT READ)
38,satisfied_democracy,9.0,Refused (DO NOT READ)
43,polsys_satisfied,8.0,Don't know (DO NOT READ)
44,polsys_satisfied,9.0,Refused (DO NOT READ)
49,fav_us,8.0,Don't know (DO NOT READ)
50,fav_us,9.0,Refused (DO NOT READ)
55,fav_china,8.0,Don't know (DO NOT READ)
56,fav_china,9.0,Refused (DO NOT READ)


In [13]:
# recode only the dictionary-defined non-response code for its matching variable
special_code_audit = []

for _, row in special_codes.iterrows():
    variable = row["variable"]
    code_value = row["value_code"]

    if variable in df.columns:
        df[variable] = pd.to_numeric(df[variable], errors="coerce")
        mask = df[variable].eq(code_value)
        replaced_count = int(mask.sum())

        if replaced_count:
            special_code_audit.append({
                "variable": variable,
                "code": code_value,
                "label": row["value_label"],
                "replaced_with_NaN": replaced_count
            })
            df.loc[mask, variable] = np.nan

special_code_audit = pd.DataFrame(special_code_audit)
print("Special response-code treatment completed.")
print("Total values recoded to missing:", int(special_code_audit["replaced_with_NaN"].sum()))
display(special_code_audit.sort_values("replaced_with_NaN", ascending=False).head(20))


Special response-code treatment completed.
Total values recoded to missing: 113559


,variable,code,label,replaced_with_NaN
19,fav_india,8.0,Don't know (DO NOT READ),3529
261,confid_macron,8.0,Don't know (DO NOT READ),3027
264,confid_netanyahu,8.0,Don't know (DO NOT READ),2870
255,confid_xi,8.0,Don't know (DO NOT READ),2844
17,fav_israel,8.0,Don't know (DO NOT READ),2747
267,confid_zelenskyy,8.0,Don't know (DO NOT READ),2620
9,fav_eu,8.0,Don't know (DO NOT READ),2542
11,fav_un,8.0,Don't know (DO NOT READ),2514
298,conflict_usparties,8.0,Don't know (DO NOT READ),2305
274,trump_personality_honest,8.0,Don't know (DO NOT READ),2163


In [14]:
# verify the important variable-specific rules after recoding
print("Remaining age code 98 (Don't know):", int((df['age'] == 98).sum()))
print("Remaining age code 99 (Refused):", int((df['age'] == 99).sum()))
print("Valid age code 97 (97 or older):", int((df['age'] == 97).sum()))
print("Q32 aireg_us code 8 retained:", int((df['aireg_us'] == 8).sum()))
assert not df['age'].isin([98, 99]).any(), 'Age special codes remain after recoding.'
assert (df['aireg_us'] == 8).sum() > 0, 'Q32 code 8 should remain a valid Not sure response.'


Remaining age code 98 (Don't know): 0
Remaining age code 99 (Refused): 0
Valid age code 97 (97 or older): 3
Q32 aireg_us code 8 retained: 3112


### Education Harmonisation


It is stated that the assessment brief explicitly requires education to be harmonised across countries. The dataset stores education in country-specific variables. Their original codes and labels are not directly comparable.

The below code uses the official data dictionary labels to create one broad common education category. The original education columns are retained. Any category that cannot be confidently grouped is marked for review rather than guessed.

In [15]:
# select the main country-specific education variables
excluded_education_variables = [
    "d_educ_germany_vocational",
    "d_educ_israel_years",
    "d_educ_israel_vocational"
]
education_variables = [
    col for col in df.columns
    if col.startswith("d_educ_") and col not in excluded_education_variables
]

education_dictionary = values_dictionary[
    values_dictionary["variable"].isin(education_variables)
].copy()
education_dictionary["value_code"] = pd.to_numeric(
    education_dictionary["value_code"], errors="coerce"
)


In [16]:
# education classification function, that harmonises country-specific education labels into broad comparable groups
def education_category(label):
    text = str(label).strip().lower()

    # normalise apostrophes, dashes and spacing
    text = (
        text.replace("’", "'")
            .replace("‘", "'")
            .replace("–", "-")
            .replace("—", "-")
    )
    text = re.sub(r"\s+", " ", text)

    # keep dictionary non-response labels separate from education categories
    if re.search(r"\bdon't know\b|\brefused\b", text):
        return np.nan

    # map labels that need a country-specific interpretation
    exact_mappings = {
        "secondary school or less": "Secondary education (broad category)",
        "bachelor and above": "Tertiary education (broad category)",
        "secondary school - long track, technical college, university entrance": "Upper secondary",
        "secondary school - short track, apprenticeship": "Vocational / technical",
        "complete preparatory": "Upper secondary",
        "senior general secondary (havo/vwo - second three grades)/hbs": "Upper secondary",
        "undergraduate education at the second level - college (brevet des collèges)": "Lower secondary",
        "gcse /cse/gce o-level (any grades) ... [see demographic qq for full wording]": "Lower secondary",
        "undergraduate qualifying program/university transfer program (cegep in quebec)": "Some tertiary / incomplete",
        "upper primary or middle school (upper primary certificate, up to 8th standard)": "Lower secondary",
        "first degree - b.a.": "Undergraduate",
        "honours degree": "Undergraduate",
        "undergraduate teaching university (diploma of general university studies (deug), license, license (lmd))": "Undergraduate",
        "technological educational institute (a.tei) (highest tertiary education), aspaite (after the anotatopoiisi)": "Undergraduate",
        "ptychio (upper tertiary certificate) ... [see demographic qq for full wording]": "Undergraduate",
        "diploma programmes (div) or program sarjana s1 (bachelors degree)": "Undergraduate",
        "tertiary complete, no university": "Vocational / technical",
        "complete post-graduation courses (specialization, professional masters, mba)": "Postgraduate",
        "program spesialis 1 (graduate diploma (si)) and profesi doktor (medical doctor diploma)": "Postgraduate",
        "post graduate diploma": "Postgraduate",
        "second degree - m.a.": "Postgraduate",
        "diploma programmes (di thru diii)": "Vocational / technical",
        "epagelmatiko lykeio (epal), epagelmatiki sxoli ... [see demographic qq for full wording]": "Vocational / technical",
        "second degree second stage ... [see demographic qq for full wording]": "Upper secondary",
        "national state schools for adults ... [see demographic qq for full wording]": "Other - review required",
        "preparation for entry to a selective formation 5b (prepa entry paramedical and social schools, classes update bts level)": "Other - review required",
        "junior college (associate degree), special training college (diploma or advanced diploma)": "Other - review required",
        "partial yeshiva": "Other - review required",
        "kolel": "Other - review required",
        "national higher certificate (nc), advanced certificate (ac)": "Other - review required",
        "national diploma": "Other - review required",
        "national diploma or national n diploma": "Other - review required",
        "advanced diploma": "Other - review required",
        "some senior high school or vocational ... [see demographic qq for full wording]": "Other - review required",
        "completed secondary school or vocational ... [see demographic qq for full wording]": "Other - review required"
    }

    if text in exact_mappings:
        return exact_mappings[text]

    # classify primary and no formal education before checking certificate or diploma wording
    if re.search(
        r"\b(no formal|no schooling|no qualifications|no educational qualifications|"
        r"did not receive formal education|has not attended school|illiterate|"
        r"pre-primary|preschool|kindergarten|primary|elementary|"
        r"less than 5 years of schooling|childhood education|early childhood|"
        r"grades 1-6|1st-7th grades)\b",
        text
    ):
        return "No formal / primary"

    # keep mixed secondary or vocational labels for review
    if re.search(r"\b(or vocational|some senior high school or vocational)\b", text):
        return "Other - review required"

    # keep incomplete secondary labels broad when the level is not specified
    if re.search(r"\bincomplete secondary\b", text):
        return "Secondary education (broad category)"

    # classify lower secondary education
    if re.search(
        r"\b(lower secondary|junior high|middle school|junior secondary|second grade junior|"
        r"incomplete junior high|incomplete middle school|grades 7-9|"
        r"grades 7-8|first three grades|brevet des collèges)\b",
        text
    ):
        return "Lower secondary"

    # classify postgraduate education
    if re.search(
        r"\b(doctorate|doctoral|doctor's degree|doctors degree|phd|"
        r"doctor degree|program doctor|master|postgraduate|post graduate|"
        r"graduate degree|graduate education|licentiate)\b",
        text
    ):
        return "Postgraduate"

    # keep incomplete higher education separate from completed degrees
    if re.search(
        r"\b(incomplete tertiary|incomplete university|"
        r"incomplete higher education|incomplete higher education course|"
        r"university transfer program)\b",
        text
    ):
        return "Some tertiary / incomplete"

    # classify completed undergraduate qualifications
    if re.search(
        r"\b(bachelor|b\.a\.|bsc|b\.sc|b\.tech|complete university|"
        r"university degree|university graduates|university/college graduates|"
        r"three-year degree|college degree|college bachelor|first degree|"
        r"university diploma|deug|licence \(lmd\)|associate's degree|"
        r"associate degree|undergraduate)\b",
        text
    ):
        return "Undergraduate"

    # classify explicitly vocational and technical qualifications
    # generic words such as certificate and diploma are not used on their own
    if re.search(
        r"\b(vocational|technical|trade|apprenticeship|apprentice|"
        r"polytechnic|professional training|professional specialization|"
        r"specialist training|advanced vocational|higher vocational|"
        r"training institute|teacher training|craft trade|industrial training|"
        r"tivet|nated|mbo|hnc|hnd|nvq|nce|technician|technologist|"
        r"technical career|commercial career|college of technology|"
        r"special training|training cycles|vocational school|"
        r"vocational training|certificate/upgrading|trade certificate|"
        r"secondary technical|secondary vocational|higher national diploma)\b",
        text
    ):
        return "Vocational / technical"

    # classify general upper secondary education after vocational pathways
    if re.search(
        r"\b(upper secondary|high school|senior secondary|secondary school|"
        r"secondary education|complete secondary|completed secondary|"
        r"a-level|a level|matriculation|baccalaureate|bachillerato|"
        r"general lykeio|school graduate|havo|vwo|"
        r"school leaving certificate|certificate of secondary education)\b",
        text
    ):
        return "Upper secondary"

    # retain labels that cannot be placed confidently for manual review
    return "Other - review required"


education_dictionary["education_harmonised"] = (
    education_dictionary["value_label"].apply(education_category)
)

display(
    education_dictionary[
        ["variable", "value_code", "value_label", "education_harmonised"]
    ]
)

print("\nDictionary mapping summary:")
display(
    education_dictionary["education_harmonised"].value_counts(dropna=False)
)

,variable,value_code,value_label,education_harmonised
2372,d_educ_argentina,1.0,No formal education (preschool or less),No formal / primary
2373,d_educ_argentina,2.0,Incomplete primary,No formal / primary
2374,d_educ_argentina,3.0,Complete primary,No formal / primary
2375,d_educ_argentina,4.0,Incomplete secondary,Secondary education (broad category)
2376,d_educ_argentina,5.0,Complete secondary,Upper secondary
...,...,...,...,...
2686,d_educ_uk,7.0,University (Bachelor's degree); undergraduate,Undergraduate
2687,d_educ_uk,8.0,University (Master's degree); post-graduate,Postgraduate
2688,d_educ_uk,9.0,"University (Doctorate, PHD)",Postgraduate
2689,d_educ_uk,98.0,Don't know (DO NOT READ),NaN



Dictionary mapping summary:


education_harmonised
No formal / primary                     63
Postgraduate                            49
NaN                                     46
Vocational / technical                  43
Upper secondary                         35
Undergraduate                           30
Lower secondary                         22
Other - review required                 11
Secondary education (broad category)     5
Some tertiary / incomplete               4
Tertiary education (broad category)      1
Name: count, dtype: int64

In [17]:
# identify each respondent's main education variable and apply its dictionary mapping
education_source = df[education_variables]
education_counts = education_source.notna().sum(axis=1)

print("Respondents with no education value:", int((education_counts == 0).sum()))
print("Respondents with multiple education values:", int((education_counts > 1).sum()))

df = df.copy()
df["education_source_variable"] = education_source.notna().idxmax(axis=1).where(
    education_source.notna().any(axis=1)
)
df["education_original_code"] = np.nan
df["education_harmonised"] = pd.Series(index=df.index, dtype="object")

for variable in education_variables:
    mask = df["education_source_variable"].eq(variable)
    codes = pd.to_numeric(df.loc[mask, variable], errors="coerce")
    mapping = education_dictionary[
        education_dictionary["variable"].eq(variable)
    ].set_index("value_code")["education_harmonised"]
    df.loc[mask, "education_original_code"] = codes
    df.loc[mask, "education_harmonised"] = codes.map(mapping)

display(df["education_harmonised"].value_counts(dropna=False))


Respondents with no education value: 129
Respondents with multiple education values: 0


education_harmonised
Upper secondary                         8453
Undergraduate                           4945
Vocational / technical                  3853
No formal / primary                     3433
Postgraduate                            3152
Lower secondary                         1955
Secondary education (broad category)    1080
Other - review required                  555
Tertiary education (broad category)      554
Some tertiary / incomplete               224
NaN                                      129
Name: count, dtype: int64

In [18]:
# count respondents whose education labels need manual review
unmapped_education = education_dictionary[
    education_dictionary["education_harmonised"].eq("Other - review required")
][["variable", "value_code", "value_label"]].copy()

unmapped_education["respondent_count"] = [
    int(pd.to_numeric(df[variable], errors="coerce").eq(code).sum())
    for variable, code in zip(
        unmapped_education["variable"], unmapped_education["value_code"]
    )
]

print("Unmapped dictionary labels:", len(unmapped_education))
print("Respondents requiring education review:",
      int(df["education_harmonised"].eq("Other - review required").sum()))
display(unmapped_education.sort_values("respondent_count", ascending=False))


Unmapped dictionary labels: 11
Respondents requiring education review: 555


,variable,value_code,value_label,respondent_count
2407,d_educ_canada,6.0,Completed secondary school or vocational ... [...,192
2535,d_educ_japan,6.0,"Junior college (associate degree), special tra...",112
2619,d_educ_safrica,13.0,National Diploma,71
2659,d_educ_sweden,5.0,National state schools for adults ... [see dem...,39
2620,d_educ_safrica,14.0,National Diploma or National N Diploma,32
2421,d_educ_france,7.0,Preparation for entry to a selective formation...,30
2406,d_educ_canada,5.0,Some senior high school or vocational ... [see...,26
2618,d_educ_safrica,12.0,"National Higher Certificate (NC), Advanced Cer...",25
2508,d_educ_israel,12.0,Kolel,16
2622,d_educ_safrica,16.0,Advanced diploma,10


### Education mapping validation

The following checks compare selected country-specific labels with their intended broad education category. Unclear labels remain in `Other - review required` for manual review.

In [19]:
# check that clear country-specific education labels have the expected category
education_mapping_checks = [
    ("d_educ_india", "Primary Education", "No formal / primary"),
    ("d_educ_india", "Upper Primary or Middle School (Upper Primary Certificate", "Lower secondary"),
    ("d_educ_india", "Senior Secondary", "Upper secondary"),
    ("d_educ_israel", "High school (NO matriculation certificate)", "Upper secondary"),
    ("d_educ_israel", "High school (WITH matriculation certificate)", "Upper secondary"),
    ("d_educ_israel", "Second degree - M.A.", "Postgraduate"),
    ("d_educ_italy", "Secondary school, second degree/high school diploma", "Upper secondary"),
    ("d_educ_italy", "No educational qualifications", "No formal / primary"),
    ("d_educ_greece", "Primary school certificate", "No formal / primary"),
    ("d_educ_greece", "3rd Certificate of Secondary Education", "Lower secondary"),
    ("d_educ_greece", "Certificate of Secondary Education / Geniko Lykeio", "Upper secondary"),
    ("d_educ_greece", "Epagelmatiko Lykeio", "Vocational / technical"),
    ("d_educ_safrica", "Complete secondary / National Senior Certificate", "Upper secondary"),
    ("d_educ_safrica", "Completed lower secondary school", "Lower secondary"),
    ("d_educ_safrica", "Incomplete secondary school", "Secondary education (broad category)"),
    ("d_educ_argentina", "Incomplete secondary", "Secondary education (broad category)"),
    ("d_educ_spain", "Certificate of Primary", "No formal / primary"),
    ("d_educ_nigeria", "Higher national diploma", "Vocational / technical"),
    ("d_educ_indonesia", "Diploma programmes (DI thru DIII)", "Vocational / technical")
]

education_check_rows = []

for variable, label_fragment, expected_category in education_mapping_checks:
    matched = education_dictionary[
        education_dictionary["variable"].eq(variable)
        & education_dictionary["value_label"].str.contains(
            label_fragment, case=False, regex=False, na=False
        )
    ]

    if len(matched) != 1:
        actual_category = "NOT FOUND OR NOT UNIQUE"
        source_label = label_fragment
    else:
        actual_category = matched["education_harmonised"].iloc[0]
        source_label = matched["value_label"].iloc[0]

    education_check_rows.append({
        "variable": variable,
        "source label": source_label,
        "expected category": expected_category,
        "actual category": actual_category,
        "status": "PASS" if actual_category == expected_category else "FAIL"
    })

education_mapping_validation = pd.DataFrame(education_check_rows)
display(education_mapping_validation)

print("Education mapping checks passed:",
      int(education_mapping_validation["status"].eq("PASS").sum()))
print("Education mapping checks failed:",
      int(education_mapping_validation["status"].eq("FAIL").sum()))

assert education_mapping_validation["status"].eq("PASS").all(), (
    "Review the education mapping function before exporting the cleaned dataset."
)

# show the final respondent counts in each harmonised education category
print("\nFinal harmonised education counts:")
display(df["education_harmonised"].value_counts(dropna=False))


,variable,source label,expected category,actual category,status
0,d_educ_india,"Primary Education (primary certificate, up to ...",No formal / primary,No formal / primary,PASS
1,d_educ_india,Upper Primary or Middle School (Upper Primary ...,Lower secondary,Lower secondary,PASS
2,d_educ_india,Senior Secondary / intermediate (Senior Second...,Upper secondary,Upper secondary,PASS
3,d_educ_israel,High school (NO matriculation certificate),Upper secondary,Upper secondary,PASS
4,d_educ_israel,High school (WITH matriculation certificate),Upper secondary,Upper secondary,PASS
5,d_educ_israel,Second degree - M.A.,Postgraduate,Postgraduate,PASS
6,d_educ_italy,"Secondary school, second degree/high school di...",Upper secondary,Upper secondary,PASS
7,d_educ_italy,No educational qualifications,No formal / primary,No formal / primary,PASS
8,d_educ_greece,Primary school certificate,No formal / primary,No formal / primary,PASS
9,d_educ_greece,3rd Certificate of Secondary Education (Junior...,Lower secondary,Lower secondary,PASS


Education mapping checks passed: 19
Education mapping checks failed: 0

Final harmonised education counts:


education_harmonised
Upper secondary                         8453
Undergraduate                           4945
Vocational / technical                  3853
No formal / primary                     3433
Postgraduate                            3152
Lower secondary                         1955
Secondary education (broad category)    1080
Other - review required                  555
Tertiary education (broad category)      554
Some tertiary / incomplete               224
NaN                                      129
Name: count, dtype: int64

### Country-specific demographic limitations

The assessment brief notes that demographic variables are not recorded identically in every country. In particular, Australia uses "age_australia" and "gender_australia" (age is banded), has no internet-use variable, and political ideology is represented using different scales in some countries. These differences are documented here rather than being combined using unsupported assumptions.

The following cell checks which relevant columns exist and how much data they contain. It does not recode or impute these variables.

In [20]:
# check the demographic fields that use different formats across countries
demographic_fields = [
    "age", "age_australia", "gender", "gender_australia",
    "political_scale2", "political_scale3", "internetfreq", "weight"
]

for variable in demographic_fields:
    if variable in df.columns:
        print(variable, ":", int(df[variable].notna().sum()), "valid values")
    else:
        print(variable, ": not available")


age : 26210 valid values
age_australia : 1827 valid values
gender : 26504 valid values
gender_australia : 1827 valid values
political_scale2 : 19011 valid values
political_scale3 : 1964 valid values
internetfreq : 26410 valid values
weight : 28333 valid values


### Final data-quality verification

In [21]:
# after all the cleaning and harmonisation steps, we are checking the final shape of the dataset and the total number of missing values to ensure that our data is in a good state for analysis
print("Final dataset shape:", df.shape)
print("Total missing values:", df.isna().sum().sum())
print("Completely duplicated rows:", df.duplicated().sum())     # cross-check duplicate rows again

Final dataset shape: (28333, 486)
Total missing values: 10767917
Completely duplicated rows: 0


### Export the cleaned dataset

The original CSV is not overwritten. The processed dataset is saved as a separate file for the Task 2 and Task 3.

In [22]:
# save the cleaned dataset for Task 2 and Task 3
df.to_csv("Task1_Cleaned_Dataset.csv", index=False)
print("Cleaned dataset saved")
print("Final dataset dimensions:", df.shape)


Cleaned dataset saved
Final dataset dimensions: (28333, 486)
